In [7]:
import polars as pl 
from pathlib import Path 
import json


COMBFOLD_OUTPUT_DIR = Path("/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/t11_RM_TM_updated_CF_pipeline/CombFold")

In [8]:
STEP_X2_OUTPUT_PATH = Path("/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/t11_RM_TM_updated_CF_pipeline/cf_pdb_structure_similarity/aggregate_cf_for_pdb_eval.parquet") 
df = pl.read_parquet(STEP_X2_OUTPUT_PATH)

In [9]:
# Filter for found_match_reference_pdb_model == True
#? See if this is necessary, as it might be that we want to keep the ones that did not find a match as well.
df = df.filter(pl.col("found_match_reference_pdb_model") == True)

In [10]:
df["CP_stochiometry"][0]

'{"P25604": 1, "P42939": 1, "Q02767": 1, "Q99176": 1}'

In [11]:
def _format_dir_name(dir_name: str):
    """
    P25604x1_P42939x1_Q02767x1_Q99176x1_pool_output is an example of a name. And they can either be type _pool or type _pair 

    We want to convert them to a dict that looks like this:
    {"P25604" : 1, "P42939" : 1, "Q02767" : 1, "Q99176" : 1}
    ie containing their proteins and multiplicities.
    """
    assert dir_name.endswith("_pool_output") or dir_name.endswith("_pair_output"), f"Directory name {dir_name} does not end with _pool_output or _pair_output"
    prot_dict = {}
    for prot in dir_name.split("_")[0:-2]:
        prot_id, count = prot.split("x")
        prot_dict[prot_id] = int(count)
    return prot_dict

# 1. Build lookup: frozenset(items) -> path
pairs = [(_format_dir_name(subdir.name), subdir) for subdir in COMBFOLD_OUTPUT_DIR.iterdir() if subdir.name.endswith("_output")]

lookup = {frozenset(d.items()): path for d, path in pairs}


# [(_format_dir_name(subdir.name), subdir) for subdir in COMFOLD_OUTPUT_DIR.iterdir() if subdir.name.endswith("_output")]

In [12]:
# 2. Parse the CP_stochiometry strings and look up the path
df = df.with_columns(
    pl.col("CP_stochiometry")
    .map_elements(
        lambda s: str(lookup.get(frozenset(json.loads(s).items()))),
        return_dtype=pl.Utf8,
    )
    .alias("Combfold_result_path")
)

In [13]:
import logging
logger = logging.getLogger(__name__)

# 3. Check the number of output_* in each {CombfoldResultDIR}/assembled_results/ subdir, and add that as a new column n_combfold_outputs
def _get_number_of_combfold_outputs(combfold_result_path) -> int:
    combfold_result_path = Path(combfold_result_path)
    assert (combfold_result_path is not None) and (combfold_result_path.is_dir()), f"Combfold result path {combfold_result_path} is not a directory"
    assembled_results_dir = combfold_result_path / "assembled_results"
    if not (combfold_result_path / "_unified_representation").is_dir():
        logger.warning(f"Combfold result path {combfold_result_path} does not contain _unified_representation subdir. Probably pair not in database.")
    if not assembled_results_dir.is_dir():
        logger.warning(f"Combfold result path {combfold_result_path} does not contain assembled_results subdir. Combfold failed to assemble.")
        return 0
    return len(list(assembled_results_dir.glob("output_*")))

df = df.with_columns(
    pl.col("Combfold_result_path").map_elements(
        lambda path: _get_number_of_combfold_outputs(path),
        return_dtype=pl.Int64,
    ).alias("n_combfold_outputs")
)

Combfold result path /cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/t11_RM_TM_updated_CF_pipeline/CombFold/P25604x1_P42939x1_Q02767x1_Q99176x1_pool_output does not contain _unified_representation subdir. Probably pair not in database.
Combfold result path /cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/t11_RM_TM_updated_CF_pipeline/CombFold/P25604x1_P42939x1_Q02767x1_Q99176x1_pool_output does not contain assembled_results subdir. Combfold failed to assemble.


In [14]:
output_dir = Path(STEP_X2_OUTPUT_PATH).parent # This is again the "cf_pdb_structure_similarity" dir
assert output_dir.is_dir(), f"Output dir {output_dir} is not a directory"
obj_cols = [c for c, dt in zip(df.columns, df.dtypes) if dt == pl.Object]
df = df.with_columns(
    pl.col(c).map_elements(lambda x: json.dumps(x) if x is not None else None, return_dtype=pl.Utf8)
    for c in obj_cols
)
df.write_parquet(output_dir / "aggregate_cf_for_pdb_eval_with_combfold_results_paths.parquet")